# 04 - 训练技巧与回调

## 学习目标
- 掌握常用回调函数
- 学会学习率调度
- 了解 TensorBoard

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import numpy as np

## 1. 回调函数

In [ ]:
# 加载数据
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 构建模型
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# 定义回调
callback_list = [
    # 早停
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    # 学习率调度
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    # 模型检查点
    callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# 训练
history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=128,
    callbacks=callback_list,
    verbose=1
)

## 2. 学习率调度策略

In [ ]:
# 自定义学习率调度
def lr_schedule(epoch, lr):
    if epoch < 5:
        return lr
    elif epoch < 10:
        return lr * 0.5
    else:
        return lr * 0.1

lr_scheduler = callbacks.LearningRateScheduler(lr_schedule, verbose=1)

# 余弦退火
cosine_scheduler = callbacks.CosineDecay(
    initial_learning_rate=0.001,
    decay_steps=1000
)

## 3. TensorBoard

In [ ]:
# TensorBoard 回调
tensorboard_callback = callbacks.TensorBoard(
    log_dir='./logs',
    histogram_freq=1,
    write_graph=True,
    write_images=True
)

# 训练（启用 TensorBoard）
model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=5,
    callbacks=[tensorboard_callback],
    verbose=0
)

print('运行: tensorboard --logdir=./logs')
print('然后访问: http://localhost:6006')

## 4. 自定义回调

In [ ]:
class CustomCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('val_accuracy') > 0.98:
            print(f'\n达到 98% 准确率，停止训练！')
            self.model.stop_training = True

# 使用自定义回调
model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=20,
    callbacks=[CustomCallback()],
    verbose=0
)

## 小结
- EarlyStopping 防止过拟合
- ReduceLROnPlateau 动态调整学习率
- ModelCheckpoint 保存最佳模型
- TensorBoard 可视化训练过程